# Projection and Residuals — All Embedding Modes

Projects Hebrew and Arabic contextual embedding spaces onto the English space
using least-squares regression, then computes the residuals.
Runs all available embedding modes in a single pass: no flag changes needed.

## What this does

For each language (Hebrew, Arabic) in each mode:
1. Find W such that `English @ W ≈ Foreign` (least-squares)
2. Compute projected: `English @ W`  — the part of Foreign explainable by English
3. Compute residual: `Foreign - projected` — what English cannot explain

The **residual** is the cross-lingual semantic signal tested in the encoding model.
By construction it is orthogonal to English (verified by the sanity checks below).

## Modes processed

| Mode             | Source     | Shape        | Notes                           |
|------------------|------------|--------------|---------------------------------|
| `sliding_window` | NB 04      | (1735, 768)  | XLM-RoBERTa, ±8-word window    |
| `contextual`     | NB 01–03   | (N, 768)     | XLM-RoBERTa, sentence context  |
| `xglm`           | NB 04b     | (1735, 2048) | XGLM-1.7B, full causal context |
| `gemmax2`        | NB 04c     | (1735, 2304) | GemmaX2-28-2B, full causal context |
| `fasttext`       | Eyal       | (1735, 150)  | Static FastText, no context     |

Modes whose input files are not found are skipped with a warning.
This means the notebook always runs to completion regardless of which
notebooks have been run upstream.

## 1. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = '../data/processed/'

# ── Mode definitions ──────────────────────────────────────────────────────────
# Each entry: (mode_name, en_file, he_file, ar_file, uses_intersection)
# uses_intersection=True means the mode has per-language index files and
# requires computing the shared word intersection before projecting.
#
MODES = [
    {
        'name'             : 'sliding_window',
        'en_file'          : DATA_DIR + 'en_sliding_window_embeddings.csv',
        'he_file'          : DATA_DIR + 'he_sliding_window_embeddings.csv',
        'ar_file'          : DATA_DIR + 'ar_sliding_window_embeddings.csv',
        'uses_intersection': False,
    },
    {
        'name'             : 'contextual',
        'en_file'          : DATA_DIR + 'en_contextual_aligned_embeddings.csv',
        'he_file'          : DATA_DIR + 'he_contextual_aligned_embeddings.csv',
        'ar_file'          : DATA_DIR + 'ar_contextual_aligned_embeddings.csv',
        'en_idx_file'      : DATA_DIR + 'en_contextual_matched_indices.csv',
        'he_idx_file'      : DATA_DIR + 'he_contextual_matched_indices.csv',
        'ar_idx_file'      : DATA_DIR + 'ar_contextual_matched_indices.csv',
        'uses_intersection': True,
    },
    {
        'name'             : 'xglm',
        'en_file'          : DATA_DIR + 'en_xglm_embeddings.csv',
        'he_file'          : DATA_DIR + 'he_xglm_embeddings.csv',
        'ar_file'          : DATA_DIR + 'ar_xglm_embeddings.csv',
        'uses_intersection': False,
    },
    {
        'name'             : 'gemmax2',
        'en_file'          : DATA_DIR + 'en_gemmax2_embeddings.csv',
        'he_file'          : DATA_DIR + 'he_gemmax2_embeddings.csv',
        'ar_file'          : DATA_DIR + 'ar_gemmax2_embeddings.csv',
        'uses_intersection': False,
    },
]

# FastText is handled separately (different input format)
FASTTEXT_FILE = '../data/processed/podcast_trilingual_embeddings.csv'

print('Modes configured:')
for m in MODES:
    print(f"  {m['name']}")
print(f'  fasttext (standalone)')

## 2. Core Functions

In [ ]:
def project_and_residual(X_foreign, X_english):
    """
    Find W such that X_english @ W ≈ X_foreign (least squares).

    Projected = X_english @ W       — part of Foreign explainable by English
    Residual  = X_foreign - Projected — what English cannot explain

    The residual is orthogonal to English by construction (least-squares
    projection theorem). The sanity check below verifies this numerically.

    Works identically for any embedding dimension (768, 2048, 150, etc.)
    """
    W, _, _, _  = np.linalg.lstsq(X_english, X_foreign, rcond=None)
    X_projected = X_english @ W
    X_residual  = X_foreign - X_projected
    return X_projected, X_residual


def mean_correlation(X, Y):
    """Mean element-wise correlation between standardised X and Y."""
    X_norm = (X - X.mean(0)) / (X.std(0) + 1e-8)
    Y_norm = (Y - Y.mean(0)) / (Y.std(0) + 1e-8)
    return float((X_norm * Y_norm).mean())


def variance_explained(residual, original):
    return 1 - np.var(residual) / np.var(original)


def load_and_intersect(mode):
    """
    Load embeddings for a mode and return (E, H, A, shared_idx).
    For modes with uses_intersection=True, computes the EN∩HE∩AR word set.
    For modes without, returns all rows with shared_idx=None.
    """
    if mode['uses_intersection']:
        en_idx = pd.read_csv(mode['en_idx_file'])['original_word_idx'].values
        he_idx = pd.read_csv(mode['he_idx_file'])['original_word_idx'].values
        ar_idx = pd.read_csv(mode['ar_idx_file'])['original_word_idx'].values

        shared = sorted(set(en_idx) & set(he_idx) & set(ar_idx))

        en_pos = {v: i for i, v in enumerate(en_idx)}
        he_pos = {v: i for i, v in enumerate(he_idx)}
        ar_pos = {v: i for i, v in enumerate(ar_idx)}

        E_full = pd.read_csv(mode['en_file']).values.astype(float)
        H_full = pd.read_csv(mode['he_file']).values.astype(float)
        A_full = pd.read_csv(mode['ar_file']).values.astype(float)

        E = E_full[[en_pos[i] for i in shared]]
        H = H_full[[he_pos[i] for i in shared]]
        A = A_full[[ar_pos[i] for i in shared]]

        print(f'  EN∩HE∩AR intersection : {len(shared)} words')
        return E, H, A, shared

    else:
        E = pd.read_csv(mode['en_file']).values.astype(float)
        H = pd.read_csv(mode['he_file']).values.astype(float)
        A = pd.read_csv(mode['ar_file']).values.astype(float)
        return E, H, A, None


print('Functions defined.')

## 3. Run All Modes

In [ ]:
results = {}   # mode_name -> {H_residual, A_residual, H_projected, A_projected,
               #               shared_idx, he_var, ar_val, shape}

for mode in MODES:
    name = mode['name']
    print(f'\n{"="*55}')
    print(f'Mode: {name}')
    print(f'{"="*55}')

    # ── Check all required files exist ────────────────────────────────────────
    required = ['en_file', 'he_file', 'ar_file']
    if mode.get('uses_intersection'):
        required += ['en_idx_file', 'he_idx_file', 'ar_idx_file']

    missing = [mode[k] for k in required if not Path(mode[k]).exists()]
    if missing:
        print(f'  SKIPPED — missing files:')
        for f in missing:
            print(f'    {f}')
        continue

    # ── Load embeddings ───────────────────────────────────────────────────────
    print('  Loading embeddings...')
    E, H, A, shared_idx = load_and_intersect(mode)

    assert E.shape == H.shape == A.shape, \
        f'Shape mismatch: EN={E.shape} HE={H.shape} AR={A.shape}'
    print(f'  Shape : {E.shape}')

    # ── Project and compute residuals ─────────────────────────────────────────
    print('  Computing projections and residuals...')
    H_proj, H_res = project_and_residual(H, E)
    A_proj, A_res = project_and_residual(A, E)

    # ── Sanity checks ─────────────────────────────────────────────────────────
    he_corr   = mean_correlation(E, H_res)
    ar_corr   = mean_correlation(E, A_res)
    he_var_ex = variance_explained(H_res, H)
    ar_var_ex = variance_explained(A_res, A)

    print(f'  Sanity — corr(EN, HE_residual) : {he_corr:.4f}  (should be ~0)')
    print(f'  Sanity — corr(EN, AR_residual) : {ar_corr:.4f}  (should be ~0)')
    print(f'  Variance explained by EN — HE  : {he_var_ex*100:.1f}%')
    print(f'  Variance explained by EN — AR  : {ar_var_ex*100:.1f}%')
    print(f'  Residual size          — HE    : {(1-he_var_ex)*100:.1f}%')
    print(f'  Residual size          — AR    : {(1-ar_var_ex)*100:.1f}%')

    # ── Save ──────────────────────────────────────────────────────────────────
    np.save(DATA_DIR + f'hebrew_residuals_{name}.npy',  H_res)
    np.save(DATA_DIR + f'arabic_residuals_{name}.npy',  A_res)
    np.save(DATA_DIR + f'hebrew_projected_{name}.npy',  H_proj)
    np.save(DATA_DIR + f'arabic_projected_{name}.npy',  A_proj)

    if shared_idx is not None:
        pd.DataFrame({'original_word_idx': shared_idx}).to_csv(
            DATA_DIR + f'contextual_shared_indices.csv', index=False
        )

    print(f'  Saved: hebrew/arabic residuals and projected ({name})')

    results[name] = {
        'shape'     : E.shape,
        'shared_idx': shared_idx,
        'he_corr'   : he_corr,
        'ar_corr'   : ar_corr,
        'he_var_ex' : he_var_ex,
        'ar_var_ex' : ar_var_ex,
    }

print(f'\nDone. Processed {len(results)} mode(s): {", ".join(results.keys())}')

## 4. FastText Residuals (Standalone)

FastText embeddings come from Eyal's independently trained static models.
Their spaces are not aligned across languages, producing a much larger residual
(~49%) than XLM-RoBERTa (~0.4%) or XGLM. This large residual reflects
the absence of explicit cross-lingual alignment, not more complementary signal.

In [ ]:
def parse_embedding(x):
    if isinstance(x, str):
        x = x.replace('[', '').replace(']', '')
        return np.array(x.split(), dtype=float)
    return np.array(x, dtype=float)


ft_path = Path(FASTTEXT_FILE)
if not ft_path.exists():
    print(f'FastText file not found — skipping:\n  {ft_path}')
else:
    print('Loading FastText embeddings...')
    ft_df = pd.read_csv(ft_path)

    E_ft = np.vstack(ft_df['en_embedding'].apply(parse_embedding).to_numpy()).astype(np.float32)
    H_ft = np.vstack(ft_df['he_embedding'].apply(parse_embedding).to_numpy()).astype(np.float32)
    A_ft = np.vstack(ft_df['ar_embedding'].apply(parse_embedding).to_numpy()).astype(np.float32)

    print(f'  EN : {E_ft.shape}')
    print(f'  HE : {H_ft.shape}')
    print(f'  AR : {A_ft.shape}')

    H_ft_proj, H_ft_res = project_and_residual(H_ft, E_ft)
    A_ft_proj, A_ft_res = project_and_residual(A_ft, E_ft)

    he_ft_var = variance_explained(H_ft_res, H_ft)
    ar_ft_var = variance_explained(A_ft_res, A_ft)

    print(f'\n  Sanity — corr(EN, HE_residual) : {mean_correlation(E_ft, H_ft_res):.4f}')
    print(f'  Sanity — corr(EN, AR_residual) : {mean_correlation(E_ft, A_ft_res):.4f}')
    print(f'  Variance explained by EN — HE  : {he_ft_var*100:.1f}%')
    print(f'  Variance explained by EN — AR  : {ar_ft_var*100:.1f}%')
    print(f'  Residual size          — HE    : {(1-he_ft_var)*100:.1f}%')
    print(f'  Residual size          — AR    : {(1-ar_ft_var)*100:.1f}%')

    np.save(DATA_DIR + 'fasttext_hebrew_residuals.npy',  H_ft_res.astype(np.float32))
    np.save(DATA_DIR + 'fasttext_arabic_residuals.npy',  A_ft_res.astype(np.float32))
    np.save(DATA_DIR + 'fasttext_hebrew_projected.npy',  H_ft_proj.astype(np.float32))
    np.save(DATA_DIR + 'fasttext_arabic_projected.npy',  A_ft_proj.astype(np.float32))

    results['fasttext'] = {
        'shape'   : E_ft.shape,
        'he_var_ex': he_ft_var,
        'ar_var_ex': ar_ft_var,
    }
    print('  Saved: fasttext_hebrew/arabic_residuals/projected.npy')

## 5. Summary Table

In [ ]:
print('\n' + '='*65)
print('RESIDUAL SUMMARY — ALL MODES')
print('='*65)
print(f'{"Mode":<18} {"Shape":<15} {"HE residual":>12} {"AR residual":>12}')
print('-'*65)

for name, r in results.items():
    shape_str = str(r['shape'])
    he_pct    = f"{(1-r['he_var_ex'])*100:.1f}%"
    ar_pct    = f"{(1-r['ar_var_ex'])*100:.1f}%"
    print(f'{name:<18} {shape_str:<15} {he_pct:>12} {ar_pct:>12}')

print('='*65)
print()
print('Interpretation:')
print('  Smaller residual = better cross-lingual alignment (XLM-RoBERTa modes)')
print('  Larger residual  = weaker alignment (FastText) or implicit alignment (XGLM)')
print('  Encoding model performance (notebook 06) determines whether')
print('  the residual actually improves brain prediction — not residual size alone.')
print()
print('Files saved per mode:')
print('  hebrew_residuals_{mode}.npy')
print('  arabic_residuals_{mode}.npy')
print('  hebrew_projected_{mode}.npy')
print('  arabic_projected_{mode}.npy')
print()
print('Next step: run 06_Encoding_Contextual.ipynb — it will loop over all modes.')

## 6. Component Budgets per Residual Block

In [ ]:
import json
from sklearn.decomposition import PCA

TARGET, CAP, K_EN = 0.90, 150, 150

def k_for(R, target=TARGET, cap=CAP):
    p = PCA(n_components=min(cap, R.shape[1])).fit(R)
    c = np.cumsum(p.explained_variance_ratio_)
    k = min(int(np.searchsorted(c, target) + 1), cap)
    return k, float(c[k-1])

manifest = {}
for mode_name in results:
    if mode_name == 'fasttext':
        H_res = np.load(DATA_DIR + 'fasttext_hebrew_residuals.npy')
        A_res = np.load(DATA_DIR + 'fasttext_arabic_residuals.npy')
    else:
        H_res = np.load(DATA_DIR + f'hebrew_residuals_{mode_name}.npy')
        A_res = np.load(DATA_DIR + f'arabic_residuals_{mode_name}.npy')

    for lang, R in [('he', H_res), ('ar', A_res)]:
        k, var = k_for(R)
        manifest[f'{mode_name}_{lang}'] = {
            'res_k': k, 'res_var_retained': round(var, 4),
            'res_native_dim': int(R.shape[1]),
            'capped': bool(k == CAP and var < TARGET),
        }
        flag = '  (CAPPED)' if manifest[f'{mode_name}_{lang}']['capped'] else ''
        print(f'{mode_name:16s} {lang}: k={k:4d}  retains {var:.1%} '
              f'of residual variance{flag}')

with open(DATA_DIR + 'block_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print('\nSaved block_manifest.json')